# 02 · Data Preprocessing

This notebook turns the raw file into the matrices used by both clustering
algorithms:

1. remove duplicates and impute missing `total_bedrooms`,
2. engineer per-household ratio features,
3. winsorise implausible ratio values,
4. select and justify the clustering feature sets,
5. standardise the features and demonstrate why scaling is essential,
6. save the processed dataset for inspection.

In [1]:
%matplotlib inline

import os
import sys
from pathlib import Path

# Avoid a noisy joblib CPU-detection warning on Windows.
os.environ.setdefault("LOKY_MAX_CPU_COUNT", "4")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Locate the project root (the folder that contains src/) so the notebooks run
# whether Jupyter was started from notebooks/ or from the project root.
ROOT = Path.cwd()
while not (ROOT / "src").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

import preprocessing as pp
import clustering as cl
import visualization as viz

viz.set_style()
pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 170)

print("Project root:", ROOT)
print("Dataset:", pp.DATA_PATH)

Project root: D:\proj\Real-Estate-Spatial-Price-Clustering\real-estate-clustering
Dataset: D:\proj\Real-Estate-Spatial-Price-Clustering\real-estate-clustering\data\housing.csv


## 1. Clean the data

`total_bedrooms` is imputed with the **median of its own `ocean_proximity`
group**. Bedroom counts differ systematically between inland and coastal block
groups, so a group-wise median is more faithful than one global median. The
overall median (435) is used only as a fallback.

In [2]:
raw = pp.load_data()
df, clean_report = pp.clean_data(raw)

print("Clean report:", clean_report)
print("Missing values remaining:", int(df.isna().sum().sum()))

Clean report: {'duplicates_removed': 0, 'rows_after_dedup': 20640, 'missing_before': 207, 'missing_after': 0}
Missing values remaining: 0


In [3]:
group_medians = raw.groupby("ocean_proximity")["total_bedrooms"].median()
pd.DataFrame({
    "group_median_total_bedrooms": group_medians.round(1),
    "global_median": raw["total_bedrooms"].median(),
})

,group_median_total_bedrooms,global_median
ocean_proximity,,
<1H OCEAN,438.0,435.0
INLAND,423.0,435.0
ISLAND,512.0,435.0
NEAR BAY,423.0,435.0
NEAR OCEAN,464.0,435.0


## 2. Feature engineering

Three ratios describe the *character* of a neighbourhood rather than its size:

| feature | formula | interpretation |
|---|---|---|
| `rooms_per_household` | `total_rooms / households` | average home size |
| `bedrooms_per_room` | `total_bedrooms / total_rooms` | bedroom share (≈ apartment density) |
| `population_per_household` | `population / households` | crowding |

In [4]:
df = pp.add_features(df)
df[["rooms_per_household", "bedrooms_per_room", "population_per_household"]] \
    .describe(percentiles=[0.01, 0.5, 0.99]).T.round(3)

,count,mean,std,min,1%,50%,99%,max
rooms_per_household,20640.0,5.429,2.474,0.846,2.581,5.229,10.357,141.909
bedrooms_per_room,20640.0,0.214,0.062,0.036,0.122,0.203,0.417,1.000
population_per_household,20640.0,3.071,10.386,0.692,1.537,2.818,5.395,1243.333


## 3. Handle implausible ratio outliers

Winsorising at the **1st/99th percentile** clips 414 values per ratio (2% of
rows) — for example, `population_per_household` drops from a maximum of 1,243 to
5.40. Raw count columns are deliberately **not** clipped: a 39,000-room block
group is a real dense district, and standardisation already limits how much any
single feature can dominate a distance.

In [5]:
before = df.copy()
df, clip_report = pp.cap_ratio_outliers(df)

pd.DataFrame(clip_report).T

,lower,upper,values_clipped
rooms_per_household,2.581133,10.357033,414.0
bedrooms_per_room,0.122469,0.416768,414.0
population_per_household,1.536686,5.394812,414.0


In [6]:
pd.DataFrame({
    "before_min": before[pp.RATIO_FEATURES].min(),
    "before_max": before[pp.RATIO_FEATURES].max(),
    "after_min": df[pp.RATIO_FEATURES].min(),
    "after_max": df[pp.RATIO_FEATURES].max(),
    "after_p99": df[pp.RATIO_FEATURES].quantile(0.99),
}).round(3)

,before_min,before_max,after_min,after_max,after_p99
rooms_per_household,0.846,141.909,2.581,10.357,10.355
bedrooms_per_room,0.036,1.000,0.122,0.417,0.417
population_per_household,0.692,1243.333,1.537,5.395,5.394


## 4. Feature selection

Two feature sets are used, because they answer different questions:

- **`geo` (2 features)** — where a block group is. This is the natural space for
  *geographical hotspots*, and distances are directly interpretable in degrees.
- **`geo_housing` (11 features)** — where it is **and** what it looks like
  (age, size, crowding, income). This is the literal "geographical and housing
  features" configuration and is tested in every notebook.

`median_house_value` is **excluded from the default clustering features** even
though it is one of the selected dataset fields. Clustering on price and then
reporting "average price by cluster" would be circular. It is used as the
interpretation target and is available in the `geo_housing_value` sensitivity
set.

In [7]:
feature_plan = pd.DataFrame([
    ("longitude", "geography", "geo, geo_housing", "west-east position"),
    ("latitude", "geography", "geo, geo_housing", "north-south position"),
    ("housing_median_age", "housing", "housing, geo_housing", "age of the housing stock"),
    ("total_rooms", "housing", "housing, geo_housing", "district size / supply"),
    ("total_bedrooms", "housing", "housing, geo_housing", "bedroom supply"),
    ("population", "housing", "housing, geo_housing", "demand / density"),
    ("households", "housing", "housing, geo_housing", "number of occupied units"),
    ("median_income", "housing", "housing, geo_housing", "strongest price correlate (r=0.69)"),
    ("rooms_per_household", "engineered", "housing, geo_housing", "home size"),
    ("bedrooms_per_room", "engineered", "housing, geo_housing", "apartment density proxy"),
    ("population_per_household", "engineered", "housing, geo_housing", "crowding"),
    ("median_house_value", "target", "interpretation only", "avoid circular clusters; tested in geo_housing_value"),
    ("ocean_proximity", "categorical", "profiling only", "already encoded by coordinates"),
], columns=["feature", "role", "used_in", "reason"])

feature_plan

,feature,role,used_in,reason
0,longitude,geography,"geo, geo_housing",west-east position
1,latitude,geography,"geo, geo_housing",north-south position
2,housing_median_age,housing,"housing, geo_housing",age of the housing stock
3,total_rooms,housing,"housing, geo_housing",district size / supply
4,total_bedrooms,housing,"housing, geo_housing",bedroom supply
5,population,housing,"housing, geo_housing",demand / density
6,households,housing,"housing, geo_housing",number of occupied units
7,median_income,housing,"housing, geo_housing",strongest price correlate (r=0.69)
8,rooms_per_household,engineered,"housing, geo_housing",home size
9,bedrooms_per_room,engineered,"housing, geo_housing",apartment density proxy


In [8]:
for name, feats in pp.FEATURE_SETS.items():
    print(f"{name:20s} ({len(feats):2d} features): {feats}")

geo                  ( 2 features): ['longitude', 'latitude']
housing              ( 9 features): ['housing_median_age', 'total_rooms', 'total_bedrooms', 'population', 'households', 'median_income', 'rooms_per_household', 'bedrooms_per_room', 'population_per_household']
geo_housing          (11 features): ['longitude', 'latitude', 'housing_median_age', 'total_rooms', 'total_bedrooms', 'population', 'households', 'median_income', 'rooms_per_household', 'bedrooms_per_room', 'population_per_household']
geo_housing_value    (12 features): ['longitude', 'latitude', 'housing_median_age', 'total_rooms', 'total_bedrooms', 'population', 'households', 'median_income', 'rooms_per_household', 'bedrooms_per_room', 'population_per_household', 'median_house_value']


## 5. Standardisation and why it is necessary

Features live on wildly different scales: `total_rooms` reaches 39,320 while
longitude spans about 10 degrees. In raw Euclidean distance, room counts alone
decide which points are "close", and geography is effectively ignored.
`StandardScaler` gives every feature mean 0 and standard deviation 1, so each
dimension contributes on equal terms.

In [9]:
features = pp.FEATURE_SETS["geo_housing"]
X, scaler = pp.build_feature_matrix(df, features)

pd.DataFrame({
    "raw_mean": scaler.mean_,
    "raw_std": scaler.scale_,
    "scaled_mean": X.mean(axis=0),
    "scaled_std": X.std(axis=0),
}, index=features).round(3)

,raw_mean,raw_std,scaled_mean,scaled_std
longitude,-119.570,2.003,-0.0,1.0
latitude,35.632,2.136,-0.0,1.0
housing_median_age,28.639,12.585,0.0,1.0
total_rooms,2635.763,2181.562,0.0,1.0
total_bedrooms,536.852,419.381,-0.0,1.0
population,1425.477,1132.435,-0.0,1.0
households,499.540,382.320,0.0,1.0
median_income,3.871,1.900,0.0,1.0
rooms_per_household,5.334,1.322,-0.0,1.0
bedrooms_per_room,0.213,0.054,-0.0,1.0


In [10]:
# Contribution of each feature to the distance between two neighbouring rows.
i, j = 0, 1
raw_i = df.loc[i, features].to_numpy(dtype=float)
raw_j = df.loc[j, features].to_numpy(dtype=float)

print(f"Raw Euclidean distance:          {np.linalg.norm(raw_i - raw_j):,.1f}")
print(f"Standardised Euclidean distance: {np.linalg.norm(X[i] - X[j]):.3f}\n")

contrib = pd.DataFrame({
    "raw_abs_diff": np.abs(raw_i - raw_j),
    "scaled_abs_diff": np.abs(X[i] - X[j]),
}, index=features)
contrib["raw_share_pct"] = (contrib["raw_abs_diff"] / contrib["raw_abs_diff"].sum() * 100).round(2)
contrib["scaled_share_pct"] = (contrib["scaled_abs_diff"] / contrib["scaled_abs_diff"].sum() * 100).round(2)
contrib.sort_values("raw_abs_diff", ascending=False).round(3)

Raw Euclidean distance:          6,706.5
Standardised Euclidean distance: 5.213



,raw_abs_diff,scaled_abs_diff,raw_share_pct,scaled_share_pct
total_rooms,6219.000,2.851,60.33,22.59
population,2079.000,1.836,20.17,14.55
households,1012.000,2.647,9.82,20.97
total_bedrooms,977.000,2.330,9.48,18.46
housing_median_age,20.000,1.589,0.19,12.59
rooms_per_household,0.746,0.564,0.01,4.47
population_per_household,0.446,0.609,0.00,4.82
median_income,0.024,0.013,0.00,0.10
latitude,0.020,0.009,0.00,0.07
longitude,0.010,0.005,0.00,0.04


The raw distance (**≈ 6,707**) is overwhelmingly produced by `total_rooms` and
`total_bedrooms`; longitude and latitude contribute almost nothing. After
standardisation the same pair is 5.21 units apart and the weight is spread
across features.

Even so, **9 of the 11 dimensions are housing attributes**: geography is only
2/11 of the combined space. Notebook 03 shows empirically that this matters for
DBSCAN.

In [11]:
X_geo, scaler_geo = pp.build_feature_matrix(df, pp.GEO_FEATURES)
print("longitude  mean / std:", round(scaler_geo.mean_[0], 3), "/", round(scaler_geo.scale_[0], 3))
print("latitude   mean / std:", round(scaler_geo.mean_[1], 3), "/", round(scaler_geo.scale_[1], 3))
print("scaled geo means:", X_geo.mean(axis=0).round(6))
print("scaled geo stds: ", X_geo.std(axis=0).round(6))

longitude  mean / std: -119.57 / 2.003
latitude   mean / std: 35.632 / 2.136
scaled geo means: [-0. -0.]
scaled geo stds:  [1. 1.]


## 6. Save the processed dataset

The cleaned, engineered frame is written to `data/housing_processed.csv` so the
result can be inspected in any spreadsheet or BI tool. Notebooks 03–05 rebuild it
deterministically through `pp.prepare_dataset()`, which guarantees that every
notebook uses identical inputs.

In [12]:
path = pp.save_processed(df)
print("saved:", path)
print("shape:", df.shape)

saved: D:\proj\Real-Estate-Spatial-Price-Clustering\real-estate-clustering\data\housing_processed.csv
shape: (20640, 13)


## Preprocessing decisions — summary

| decision | choice | reason |
|---|---|---|
| Missing `total_bedrooms` (207 rows) | median of `ocean_proximity` group | preserves coastal/inland differences; 1% of data |
| Duplicates | remove | none found, but keeps the pipeline safe |
| Ratio outliers | winsorise at 1st/99th percentile | stops non-residential records from dominating distances |
| Raw counts | keep | large values are genuine dense districts |
| Price cap | keep capped rows | top-coded, not erroneous; they are the expensive coast |
| Feature sets | `geo` and `geo_housing` | separate "where" from "where + what" |
| Price in clustering | excluded by default | avoids circular interpretation; tested separately |
| Scaling | `StandardScaler` | makes Euclidean distances dimension-neutral |